# Quantum Approximate Multi-Objective Optimization for constrained optimization problems (C-QAMOO)

Reference paper: [Binary Optimization with Complex Constraints
via Quantum Approximate Multi-Objective Optimization](https://arxiv.org/pdf/2609.13790)

Challenge author: Ibrahim Shehzad

Quantum approximate multi-objective optimization (QAMOO) is a promising application of quantum computing due to its potential to outperform classical approaches while only requiring near-term hardware friendly quantum circuits. QAMOO determines an approximate solution to a multi-objective optimization (MOO) problem. 

The need to do multi-objective optimization is something we encounter in our daily lives. For example, when we are traveling, there are multiple objectives such as cost (e.g. fuel consumption, ticket prices), distance, duration, and ecological footprint, that need to be optimized in tandem.
However, it is typically not possible to optimize all objectives simultaneously: an improvement in the value of one objective may cause the value of another objective to deteriorate (e.g. cheaper tickets often impose a longer travel time).

The goal of MOO therefore is not to compute a single solution but rather a set of points $x \in P\subset X$ that define a [Pareto front](https://en.wikipedia.org/wiki/Pareto_front), given by (written in a minimization form):

$$
PF = \left\{ (f_1(x), \dots, f_m(x)) \;: x \in X : \;
\nexists x' \in X \;\text{such that}\;
\forall i\; f_i(x') \leq f_i(x) \;\wedge\;
\exists j\; f_j(x') < f_j(x)
\right\}
$$


At a conceptual level, it is the boundary of the achievable objective space where you cannot improve one objective without worsening at least one other objective.

In this challenge, we will show how one can make use of QAMOO to solve a constrained optimization problem. 

#### C-QAMOO on a 100 asset investment portfolio

The specific problem we will consider in this challenge is a constrained optimization problem on an investment portfolio of 100 assets, where we want to maximize the expected return and restrict the [Conditional Value at Risk (CVaR)](https://en.wikipedia.org/wiki/Expected_shortfall) of the loss, to be less than a desired value. Stated informally, the CVaR, which is parametrized by a confidence value $\alpha$, is the expectation value of the loss on the worst $1-\alpha$ fraction of the outcomes. It is intuitive to see why in a healthy investment portfolio we would want to minimize that.

Let $R \in \mathbb{R}^n$ denote a random vector of asset returns,
and assume $ 
R \sim \mathcal{N}(\mu, \Sigma)$ (i.e., asset returns from the portfolio follow a joint Gaussian distribution)
where $\mu \in \mathbb{R}^n$ is the vector of expected returns and
$\Sigma \in \mathbb{R}^{n \times n}$ is the covariance matrix.
Binary decision variables $x \in \{0,1\}^n$ indicate asset selection,
where $x_i = 1$ means that asset $i$ is included in the portfolio.
The corresponding portfolio return and loss are $R^\mathsf{T}x$ and $L(x) = -R^\mathsf{T}x$ respectively. 

The optimization problem in question can be formulated as

\begin{aligned}
\max_{x \in X} \quad & \mu^\mathsf{T}x \\
\text{s.t.} \quad & \operatorname{CVaR}_{\alpha}(L(x)) \leq c \,,
\end{aligned}

for some constant c. As shown in the [paper](https://arxiv.org/pdf/2609.13790), this can be written as

\begin{aligned}
\max_{x \in X} \quad & \mu^\mathsf{T}x \\
\text{s.t.} \quad & -\mu^\mathsf{T}x
+ k_\alpha \sqrt{x^\mathsf{T}\Sigma x}
\leq c \,
\end{aligned}

where $k_\alpha$ is a constant that depends on the confidence level $\alpha$. For our values of interest in this challenge, $k_{\alpha = 0.99} = 2.6652$ , $k_{\alpha = 0.95} = 2.0627$, and $k_{\alpha = 0.90} = 1.7550$.

The preferred algorithm for optimization using quantum computers is the Quantum Approximate Optimization Algorithm (QAOA) which is a hybrid quantum-classical algorithm that can be used to solve combinatorial optimization problems. The QAOA is a variational algorithm that uses a parameterized quantum circuit to prepare a quantum state that encodes the solution to the optimization problem. The parameters of the circuit are optimized using classical optimization techniques, with the goal of minimizing the expected value of the objective function. QAOA was initially developed for unconstrained optimization. While it can be augmented to deal with constraints, this typically comes at a cost which translates into deeper circuits and more parameters to optimize. This generally leads to sub-optimal performance on near-term quantum hardware. 

The main novelty of this paper is a new algorithm for solving constrained binary optimization problems, that is well suited for QAOA implemented on near-term hardware. For the specific problem described above, this algorithm translates into solving for the Pareto front of the following unconstrained multi-objective optimization problem (see the accompanying paper for details): 

1. $min_{x \in X} \{-\mu^{T} x, x^{T} \Sigma x \}$, 
   and then
2. Postselecting the points on the pareto front that satisfy the CVaR constraint.



#### Sparsifying the covariance matrix: Adapting to the quantum hardware
The full covariance matrix couples all 100 assets to each other, which is
$\binom{100}{2} = 4{,}950$ pairs, and the corresponding quantum circuit would incur a 2-qubit gate for every pair of coupled assets it
keeps. To adapt this to hardware with limited connectivity (like the IBM quantum hardware), the matrix is sparsified: only the pairs (qubits) that are connected (with a controlled application of swap gates) are kept, and every other entry of the covariance matrix is set to $0$.

The reason why the application of swap gates is kept controlled is as follows: since swap gates in general lead to deeper circuits and noisier circuit executions on the hardware, we restrict the number of swap layers that the routed circuit can use. This number, the *sparsity*, is denoted by $K$ and here, we restrict $0 \leq K \leq 4$, where $K = 0$ leads to the sparsest covariance matrix. Our goal will be to show that sparsifying the covariance matrix gives us healthy portfolios, while also allowing us to run the circuits on near-term quantum hardware.



#### The Weighted Sum Method (WSM)

A core building block of QAMOO is the Weighted Sum Method (WSM). The WSM involves transforming a multi-objective optimization problem into a single-objective optimization problem by assigning weights to each objective function and combining them into a single objective function:
\begin{equation}
f(x)= \sum_{i=1}^{m} w_{i} f_{i}(x)\,.
\end{equation}

Adapted to our problem of interest, WSM is implemented as follows. The 2 objectives are collapsed into 1 function $f$ with weights $a$ and $b$ for the return and variance parts, respectively:

\begin{equation}
f(x) \;=\; a\,\bigl(-\mu^{\mathsf{T}} x\bigr) \;+\; b\,\bigl(x^{\mathsf{T}} \Sigma x\bigr)\,.
\end{equation}

Here: 
- $x \in \{0, 1\}^{100}$ denotes the choice of assets, with there being 1 binary variable per asset.
- The minus sign with the $\mu^{\mathsf{T}}$ term ensures that the whole of $f$ is to be minimized.

$f$ is our overall objective function. One pair $(a, b)$ is 1 ordinary single-objective problem, and its solution is
1 point in the return-variance trade-off plane, whose Pareto front we wish to calculate. 

Note that the 2 parts of $f$ are not the same size. Over the various choices of assets, the return
part $-\mu^\top x$ runs from $0$, the empty choice, to $-\sum_i \mu_i$, the all-ones
choice (every $\mu_i$ here is positive); the variance part $x^\top \Sigma x$
runs from $0$ to $\mathbf{1}^\top \Sigma \mathbf{1}$, because $\Sigma$ here has no
negative entry. A simple weighted sum of 2 parts with different spans would let 1 of
them dominate at every weight setting. To work around this, each part is divided by its own span first:

$$
a \;=\; \frac{w_{\text{return}}}{\mu^{T} \mathbf{1}} ,
\qquad
b \;=\; \frac{w_{\text{variance}}}{\mathbf{1}^\top \Sigma \mathbf{1}},
\qquad
f(x) \;=\; a\,\bigl(-\mu^\top x\bigr) + b\,\bigl(x^\top \Sigma x\bigr).
$$

In addition, both $\mu^{T}x$ and $x^{T} \Sigma x$ are offset so that their minimum value is $0$. This is done by subtracting off the minimum value of each part.



*Aside*: The diagonal shift to restore positive semi-definiteness

Setting terms in the covariance matrix to $0$ costs the matrix a property that it is required to have. A
covariance matrix is supposed to be positive semidefinite, $x^\top \Sigma x \ge 0$ for every $x$,
which is what makes the variance of the selection $x$ a meaningful quantity.
Sparsifying it can break this property. To amend this, 1 scalar is added to the entire
diagonal,

$$
\Sigma \;=\; \Sigma_{\text{masked}} \;+\; \delta\, I,
\qquad
\delta \;=\; \max\Bigl(0,\; -\lambda_{\min}\bigl(\Sigma_{\text{masked}}\bigr) \;+\; \varepsilon\Bigr),
\qquad
\varepsilon \;=\; 10^{-8}.
$$

which restores positive semi-definiteness. Where the sparsified matrix is still positive definite, which in this instance is the case at $K = 0$ and $K = 1$, no shift is needed and $\delta = 0$. Where it is not, the shift changes the variance scale, and therefore $b$, while $a$ is unchanged. 

*Note*: In the weights imported in this challenge, this shift as well as the aforementioned rescaling are already done.

#### Where the quantum part (actually) enters

The discussion so far was purely classical. To use a quantum computer to solve this problem, we proceed as follows. Recall that $f$ is quadratic in binary variables. Substituting $x_i = (1 - z_i) / 2$, with a spin variable
$z_i \in \{-1, +1\}$, (and promoting $z_i$ to the Pauli $Z_{i}$) converts this into an Ising Hamiltonian:

$$H_{c}(z) \;=\; \sum_{i} h_i Z_i \;+\; \sum_{i<j} J_{ij} Z_i Z_j \;+\; \text{offset}\,$$

where:
- $h_i$ is the magnetic field at site $i$. It encodes the expected return of asset $i$ and the
  diagonal of the covariance matrix.
- $J_{ij}$ is the coupling constant between the pair $(i, j)$, and it encodes the covariance of the 2 assets, $i,j$.
- $\text{offset}$ is a constant, and it absorbs all the constant terms that come from substituting $x_i = (1 - z_i) / 2$ into $f$.
It does not play a part in the optimization, but it is needed to recover the value of $f$ from the value of the hamiltonian.

Minimizing $H_{c}$ is the same as minimizing $f$. We use QAOA to do this minimization, where our problem is represented by a circuit on 100 qubits, 1 qubit per asset. The cost layer has the standard $e^{-i \gamma_\ell H_c}$ while the mixer layer has the standard $e^{-i \beta_\ell H_M}$, with $H_M = -\sum_{i} X_i$, for layers $\ell = 0, \dots, p-1$. Note that the cost layer is where the objective is written into the gates, so $a$ and $b$ (and through them the weights) end up inside the `rz` and `rzz` gates.



### To start, setup a new virtual environment and install the required packages.

In addition to Jupyter, you also need to set up your IBM Cloud account if you haven't yet.

1. Create a new python (Python 3.12–3.14) environment and activate it.

2. Install required packages from the `requirements.txt` file.

Set `TEAM_NAME` below and run the cell before you grade anything. Every member must type the team name exactly the same way (capitals and spaces count). It can have letters, digits, spaces, hyphens and underscores, up to 35 characters. Also, your team's best score on each exercise counts, so you can resubmit.

In [ ]:
from qc_grader.challenges.qdc_2026 import join_team

TEAM_NAME = ""  # YOUR TEAM NAME, the same for every member

if TEAM_NAME:
    join_team(TEAM_NAME)
else:
    print(
        "Set TEAM_NAME to your team's name and run this cell again: answers are only graded for a team."
    )

<div class="alert alert-block alert-success">

### **Your Task:**
### **Use (Linear-Ramp) QAOA to compute the Pareto front of the return-variance trade-off for a specific 100-asset investment portfolio, then impose the CVaR constraint and read off the best return available under a specified risk budget.**
The instance is fixed: 100 assets, 1 qubit per asset, with the covariance matrix sparsified to the pairs that a swap network of $K$ layers can reach.

### Exercise 1.1 (Loading the (provided) instance and mapping it to circuits):
- Sparsity levels $K = \{0, 2, 4\} \longrightarrow \{99, 198, 297\}$ retained pairs.
- QAOA depths $p = \{1, 2, 3\}$.
- 1 parametric circuit per $(p, K)$, with the 2 objective multipliers and the $2p$ angles left free, so 1 transpilation serves every weight setting.
- Trained QAOA parameters given for $K = 4$ at $p = 3$, the combination that runs on hardware, so no training is required. The other levels and depths are compared by circuit structure only.

### Exercise 1.2 (Optimizing the circuits for quantum execution):
- A chain of 100 connected qubits with the best layer fidelity on a backend, specified for the circuit layout.
- A swap network inserted *before* transpilation, with `ElidePermutations` removed so that it survives transpilation.
- Weight settings: 20 (`weight_indices = range(0, 100, 5)`).
- Only $p = 3$ at $K = 4$ is submitted (`RUN_DEPTHS = (3,)`, `RUN_LEVELS = (4,)`), at a number of shots you choose.

### Exercise 2.1 (Analyzing the QPU execution):
- The Pareto front over **every** shot.
- Priced by hypervolume against the instance-fixed corner $(\text{return}, \text{variance}) = (0, \mathbf{1}^{\mathsf{T}} \Sigma \mathbf{1})$.
- Compared against 3 classical references: $\varepsilon$-constrained CPLEX, weighted-sum CPLEX, and a random control.
- CVaR confidence levels $\alpha = \{0.90, 0.95, 0.99\}$, risk budget $0 \leq c \leq 10$, compared at $c = 1$
- Your run is graded against a reference run. To get the most out of it, consider an appropriate error suppression setting, or even your own parameter training.

### Exercise 2.2 (Going deeper [and beyond the paper]):
- At $K = 1$, trained $p = 3$ angles are provided, at 3 weight settings.
- Train $p = 4$ yourself at the same 3 weight settings, and run it.
- Your $p = 4$ run is compared against the reference $p = 3$ run, on hypervolume and on the best return under a CVaR budget of $c = 0.1$ at $\alpha = 0.9$.
- The angles come from **your own training**.
- Check what the extra depth buys.

</div>

<div class="alert alert-block" style="background-color:#1b5e20; color:#ffffff; border-color:#1b5e20;">

### **Exercise 1.1; Points [30/100]**

</div>

#### Loading in and sanity checking the portfolio instance 

Let us now load in the portfolio instance, select the sparsity levels for the covariance matrices, and the qaoa depths. We pick sparsity levels $K = 0, 2, 4$ and qaoa depths $p = 1, 2, 3$. Trained angles are provided for $K = 4$ at $p = 3$ only, which is the combination we run on hardware (`RUN_LEVELS`, `RUN_DEPTHS`). The other levels and depths are used to compare the circuits they lead to in terms of gate counts and depth. 

In [ ]:
import src
import numpy as np

DEPTHS = (1, 2, 3)
sparsities = (0, 2, 4)
# The 1 combination with trained angles, and the one that runs on hardware.
RUN_DEPTHS = (3,)
RUN_LEVELS = (4,)
portfolios = {level: src.Portfolio(level) for level in sparsities}

The `portfolios` dictionary holds key, value pairs, where the key is the sparsity level and the value is the portfolio (at the specified sparsity level). Using the attribute `portfolio.n` for each portfolio, assert check that each portfolio has 100 assets.

In [ ]:
## YOUR CODE HERE

The two defining features of each portfolio are 1. the expected returns and 2. the covariances of the assets. For the portfolio to be realistic, the expected returns should be positive and the covariance matrix positive semi-definite. In our setup, the vector of expected returns is given by `portfolio.mu` and the matrix of covariances by `portfolio.sigma`. Check if both variables obey the aforementioned properties.

In [ ]:
## YOUR CODE HERE

As the sparsity is increased, we allow more pairs of assets to be correlated. This is encoded in the `portfolio.retained_pairs` attribute. Print the list of the number of retained pairs for each sparsity level and assert that the number of retained pairs is increasing with the sparsity level.

In [ ]:
## YOUR CODE HERE

#### Aside: Parameter training

When describing QAOA in the introduction, we mentioned that the starting point of QAOA is a parametrized ansatz whose parameters are fixed after training. While this training can be done on the quantum hardware, it is often beneficial to do it classically (whenever possible). This is a fascinating area of research and the interested reader is referred to [this paper](https://arxiv.org/abs/2606.05311) for a survey of best practices and techniques for learning these parameters. Suffice it to say that the findings of this paper served as motivation to use Pauli Propagation methods (see, e.g, [this addon](https://github.com/Qiskit/pauli-prop) and [this script](https://github.com/qiskit-community/qaoa_training_pipeline/blob/main/qaoa_training_pipeline/evaluation/pauli_propagation.py)) to do the classical training for the problem at hand. As described in the [paper](https://arxiv.org/pdf/2609.13790), this involves extensive classical compute on a cluster. The trained parameters for $K = 4$ at $p = 3$, the combination run on hardware, are provided in `data/angles/angles_p3.csv`.  

The technique used to train these parameters falls under [LINEAR RAMP QAOA](https://arxiv.org/abs/2405.09169) (LR-QAOA) (see, e.g, [this script](https://github.com/qiskit-community/qaoa_training_pipeline/blob/main/qaoa_training_pipeline/training/lrqaoa_trainer.py) for LR-QAOA based training) whereby the parameters are determined by just 2 numbers $\Delta \beta$ and $\Delta \gamma$, for every weight setting:

$$\beta_\ell = -\Bigl(1 - \frac{\ell}{p}\Bigr) \Delta\beta,
\qquad
\gamma_\ell = \frac{\ell + 1}{p} \cdot \frac{\Delta\gamma}{\kappa},
\qquad \ell = 0, \dots, p-1 \,.$$

$\Delta\beta$ is the mixer slope, $\Delta\gamma$ the cost
slope, and $\kappa$ is the scale of the cost operator at that weight
setting, defined as:

$$\kappa = \max\Bigl( \max_{i} |h_i|, \;\; \max_{i<j} |J_{ij}| \Bigr).$$

Dividing by $\kappa$ normalizes the cost operator so its largest coefficient is 1. This puts $\Delta\gamma$ on a comparable scale at every weight setting, so a single search range (grid) for $\Delta\gamma$ can be used when training all of them, even though the optimal $\Delta\gamma$ still differs from one weight setting to the next.


#### Mapping to quantum circuits

In the discussion above, we described how to solve this optimization problem using a quantum computer, we convert it into a Ising hamiltonian. In the next cell, you are asked to do this conversion. You might find use for the tools available in [Qiskit addon optimization mapper](https://github.com/qiskit/qiskit-addon-opt-mapper) for doing this.

In [ ]:
problem = portfolios[0] # test portfolio
a=1.0 # test weights
b=1.0 # test weights

model, _ = src.weighted_sum_model(problem, a, b) 

## Convert model into quadratic problem
problem_op = ## YOUR CODE HERE
## Construct the QUBO
qubo = ## YOUR CODE HERE

# Construct the ising hamiltonian (and the constant, offset term) from the QUBO
# Construct this as a `SparsePauliOp` object from `qiskit.quantum_info`
hamiltonian, offset = ## YOUR CODE HERE

A sanity check of this hamiltonian is that all terms should be "pure" Z terms and at most weight 2. Check this by filling out the two functions below.

In [ ]:
def is_pure_z(hamiltonian):
    """Return true when every term of `hamiltonian` is built from I and Z alone."""
    ## YOUR CODE HERE


def is_at_most_weight_2(hamiltonian):
    """Return true when no term of `hamiltonian` acts on more than 2 qubits."""
    ## YOUR CODE HERE

<div class="alert alert-block alert-info">

### **Check your mapping to an Ising hamiltonian below (5 pts total).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_1a

grade_lab_qamoo_ex1_1a(is_pure_z, is_at_most_weight_2, hamiltonian);

Now that you have practice with the mapping, in the rest of the challenge this mapping will be done for you under the hood, so that you can focus on other parts of the challenge.


The performance of quantum algorithms rests crucially on the mapping to quantum circuits. Not all mappings are created equal and a good mapping can make a huge difference in the quality of the results. The [Quantum optimization best practices repository](https://github.com/qiskit-community/qopt-best-practices) carries a set of tools for mapping quantum optimization problems to quantum circuits as efficiently as possible. These are transpilation passes that are used in tandem with other passes available in Qiskit. Let us see how these tools effect the mapping of our problem to quantum circuits. We will use them to map one cost layer of our circuit and compare the subsequent circuit depth with a more naively transpiled circuit.

In [ ]:
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import (
    Commuting2qGateRouter,
    SwapStrategy,
)
from qopt_best_practices.transpilation.prepare_cost_layer import PrepareCostLayer
from qopt_best_practices.transpilation.swap_cancellation_pass import (
    SwapToFinalMapping,
)
from qiskit.transpiler import PassManager


def route_cost_layer(layer, n: int):
    """Route 1 cost layer onto a line of n qubits with a swap network."""

    # Build a coloring map of qubits on a chain.
    # NOTE: To get the most optimal transpiled depths
    # color such that that layers with "odd" links ((1,2), (3,4) etc.)
    # come before the ones with "even" links ((2,3), (4,5)etc.)
    # coloring = ## YOUR CODE HERE
    manager = PassManager(
        [
            PrepareCostLayer(),
            Commuting2qGateRouter(SwapStrategy.from_line(list(range(n))), coloring),
            SwapToFinalMapping(),
        ]
    )
    return manager.run(layer)

<div class="alert alert-block alert-info">

### **Check your edge colouring below (10 pts total).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_1b

grade_lab_qamoo_ex1_1b(route_cost_layer, portfolios);

Let us now build the cost layer.

In [ ]:
from src import cost_operator

## Use a parametric expression for `cost_angle`
cost_angle = ## YOUR CODE HERE

## Assign `problem` to the portfolio at some (any) value of sparsity.
## Remember the possible values of sparsity are 0,2, and 4. Passing these
## in as keys to the `portfolios` dictionary, you get a portfolio.
problem = ## YOUR CODE HERE

weights = (1, 1)  # random choice

Let's now build one cost layer of the QAOA circuit and see how its (pre-transpilation) depth behaves with and without the passes defined in `route_cost_layer` above.

Fill in the code block below. You can write your own code or look up the provided cheatsheet for libraries that have functions for doing this.

In [ ]:
def cost_layer(problem, weights, theta):
    "One unrouted cost layer"

    cost_op = cost_operator(problem, weights)
    layer = ## YOUR CODE HERE
    free = [item for item in layer.parameters if not item.name.startswith("cW[")]
    if len(free) != 1:
        raise RuntimeError(
            f"expected 1 cost angle, found {[item.name for item in free]}"
        )
    return layer.assign_parameters({free[0]: theta}, inplace=False)


circ_naive = cost_layer(problem, weights, cost_angle)

## Print the 2 qubit gate depth of this circuit
## YOUR CODE HERE

circ_informed = route_cost_layer(circ_naive, problem.n)

## Print the 2 qubit gate depth of this circuit and compare with the depth calculated
## above
## YOUR CODE HERE

<div class="alert alert-block alert-info">

### **Check your cost layer build and routing below (5 pts).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_1c

grade_lab_qamoo_ex1_1c(problem, circ_naive, circ_informed);



Let us now setup our QAOA circuits


In [ ]:
def build_parametric(problem, p: int, pre_swap):

    from qiskit.circuit import Parameter, ParameterVector

    num_qubits = problem.n
    weights = ParameterVector("cW", 2)

    cost_angle = Parameter("__theta_cost")

    if pre_swap:
        # 1 cost layer, routed once by the swap network
        routed = route_cost_layer(cost_layer(problem, weights, cost_angle), num_qubits)
        p2l, _ = src.route_map(routed)
    else:
        routed = cost_layer(problem, weights, cost_angle)
        p2l = np.arange(num_qubits, dtype=np.int64)

    betas = ParameterVector("beta", p)
    gammas = ParameterVector("gamma", p)

    ## Set up quantum circuit, called `circuit` with a quantum register
    ## and a classical register, both of size `num_qubits`
    ## Name the quantum register "q" and the classical register "meas"
    ## YOUR CODE HERE

    ## Initialize all qubits in a normalized superposition of '0' and '1'.
    ## YOUR CODE HERE

    for layer_index in range(p):
        layer_circuit = routed.assign_parameters(
            {cost_angle: gammas[layer_index]}, inplace=False
        )
        # Add layer_circuit to `circuit`.
        #  For odd layers add it in reverse order to use swap cancellations.
        ## YOUR CODE HERE

        ## Insert the qaoa mixer
        ## YOUR CODE HERE

    if p % 2 == 1:
        for physical, logical in enumerate(p2l.tolist()):
            circuit.measure(int(physical), int(logical))
    else:
        for qubit in range(num_qubits):
            circuit.measure(qubit, qubit)

    return circuit

<div class="alert alert-block alert-info">

### **Check your `build_parametric` implementation below (10 pts total).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_1d

grade_lab_qamoo_ex1_1d(build_parametric, portfolios);

The parameters and weights are read based on the weight indices. To keep our quantum execution time contained, we will only run the experiments for 20 weight settings. 

In [ ]:
weight_indices = range(0, 100, 5)

To run our job on a quantum computer, let us now retrieve a backend.

NOTE: Using fractional gates can give you non-optimally transpiled circuits for this problem which can cost you points below.

In [ ]:
# Instantiate QiskitRuntimeService
# """
# service = QiskitRuntimeService(
#     channel  = channel,
#     instance = instance,
#     token    = token )
# """

backend = service.backend("<...>")
print(f"Using backend {backend.name}")

<div class="alert alert-block" style="background-color:#1b5e20; color:#ffffff; border-color:#1b5e20;">

### **Exercise 1.2; Points [20/100]**

</div>

#### Optimizing the circuits for quantum execution

Let's first develop some appreciation for how much difference an informed transpilation can make. First, we transpile our circuits by using a "naive" transpilation, whereby no preferred layout is picked and no pre-transpilation swap networks are inserted. 

In [ ]:
# Instantiate a preset pass manager that transpiles to the backend,
# fix a seed for reproducibility.
## pm_naive = YOUR CODE HERE
built = {}
for p in DEPTHS:
    for level, one in portfolios.items():
        parametric = build_parametric(one, p, pre_swap=False)
        built[(p, level)] = {"circuit": parametric}

weight_list = list(weight_indices)
pub_levels = list(RUN_LEVELS)


def parameter_rows(isa, p: int, level: int) -> np.ndarray:
    par_names = [parameter.name for parameter in isa.parameters]
    qaoa_slopes = src.load_slopes(p, level)  # the LR-QAOA slopes of this depth
    values = np.empty((len(weight_list), len(par_names)), dtype=np.float64)
    for slot, instance in enumerate(weight_list):
        row = qaoa_slopes[instance]
        a, b = src.trained_weights(row)  # the 2 weights/multipliers
        betas, gammas = src.trained_angles(row, p)  # the 2p angles
        named = {"cW[0]": float(a), "cW[1]": float(b)}
        for layer in range(p):
            named[f"beta[{layer}]"] = float(betas[layer])
            named[f"gamma[{layer}]"] = float(gammas[layer])
        values[slot] = [
            named[name] for name in par_names
        ]  # parameters assigned in lexical order following qiskit conventions
    return values


isa_naive = {key: pm_naive.run(item["circuit"]) for key, item in built.items()}


print(f"{'p':>3} {'K':>3} {'2q depth':>9} {'2q gates':>9}")
for p in DEPTHS:
    for level in sparsities:
        isa = isa_naive[(p, level)]
        print(
            f"{p:>3} {level:>3} {isa.depth(lambda x: len(x.qubits) == 2):>9} "
            f"{isa.num_nonlocal_gates():>9}"
        )

# Trained angles exist for the run combination only, so only it becomes a PUB.
pubs = {}
for p in RUN_DEPTHS:
    pubs[p] = []
    for level in pub_levels:
        isa = isa_naive[(p, level)]
        values = parameter_rows(isa, p, level)
        pubs[p].append((isa, values))

<div class="alert alert-block alert-info">

### **Check your naive transpilation below (5 pts).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_2a

grade_lab_qamoo_ex1_2a(pm_naive, backend);

At such large values of depth, the quantum execution on existing quantum devices is unlikely to give good results. Let's instead do a more "informed" transpilation: let's transpile to a chain of qubits with a swap network inserted pre-transpilation

Here, we will pick a chain of 100 qubits with the best layer fidelity on the backend to transpile to. In addition, we will insert a swap network before the transpilation. For this, we set the flag `pre_swap = True` which invokes passes from the [Quantum optimization best practices repository](https://github.com/qiskit-community/qopt-best-practices). To ensure that the subsequent transpilation to the hardware does not undo these swap networks, we will need to remove the `ElidePermutations` pass from the qiskit transpiler.

As you should see, this "alternative" transpilation gives us a reduction in the 2-qubit gate count and depth, which is important for the circuit's fidelity on hardware. 



Let's start by obtaining the best 100-qubit chain from the hardware.

In [ ]:
## Pulling the most up to date from the backend,
# retrieve the 100 qubit chain with the best Layer fidelity.
## current_time = YOUR CODE HERE
## best_100q_chain = YOUR CODE HERE

<div class="alert alert-block alert-info">

### **Check your 100-qubit chain below (10 pts).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_2b

grade_lab_qamoo_ex1_2b(best_100q_chain, backend);

Let us visualize the chain that we picked out above.

In [ ]:
# Write code using qiskit.visualization to show the chain of selected qubits
# on the device coupling map

# NOTE: For this, you may need graphviz which is not pip installable
# and requires a system level installation. If you don't have it
# would rather not install it, skip this cell: it isn't graded.

## YOUR CODE HERE

Now instantiate a pass manager to transpile to the backend, exclude the `ElidePermutations` pass, and observe the circuit depth and 2-qubit gate count.

In [ ]:
# Instantiate a new pass manager that always transpiles to the best-100-qubit-chain
# obtained above.

# pm_informed = YOUR CODE HERE

# Remove the ElidePermutations pass from `pm_informed`
## YOUR CODE HERE

built = {}
for p in DEPTHS:
    for level, one in portfolios.items():
        # Set pre_swap = True to use the swap network discussed above.
        parametric = build_parametric(one, p, pre_swap=True)
        built[(p, level)] = {"circuit": parametric}

isa_informed = {key: pm_informed.run(item["circuit"]) for key, item in built.items()}

## Check if the final layout of the transpiled circuits is the same as the
## best-100-qubit-chain that we picked.
## YOUR CODE HERE

print(f"{'p':>3} {'K':>3} {'2q depth':>9} {'2q gates':>9}")
for p in DEPTHS:
    for level in sparsities:
        isa = isa_informed[(p, level)]  # parametrized, transpiled circuits.
        print(
            f"{p:>3} {level:>3} {isa.depth(lambda x: len(x.qubits) == 2):>9} "
            f"{isa.num_nonlocal_gates():>9}"
        )

pubs = {}
for p in RUN_DEPTHS:
    pubs[p] = []
    for level in pub_levels:
        isa = isa_informed[(p, level)]
        values = parameter_rows(isa, p, level)  # trained angles
        # To speed up our hardware execution, instead of submitting bound circuits, we
        # will submit parametrized circuits along with the relevant angle bindings.
        # Set up the PUBS for this and append each pub to pubs[p]. See
        # https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output

        ## YOUR CODE HERE

<div class="alert alert-block alert-info">

### **Check your pinned-layout transpilation below (5 pts).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex1_2c

grade_lab_qamoo_ex1_2c(isa_informed, best_100q_chain);

As you can see, the 2q depths and the number of 2q gates are now significantly reduced. Therefore, we will use this transpilation method for the rest of the experiments.

Before moving on, re-run the code above with `pre_swap = False` and compare the circuit depth and 2-qubit gate count with the previous run. You should see that the pre-swap network reduces both the depth and 2-qubit gate count of the transpiled circuits. Can you reason why this is the case?

Let us now instantiate the Sampler which is the primitive we will use to run the circuits on the quantum hardware. 

In [ ]:
## options = ## YOUR CODE HERE
## sampler = ## YOUR CODE HERE

####  Execution: Running the QAOA circuits on the quantum hardware

The performance of QAOA is expected to improve with the number of layers $p$. However, the circuit depth and the number of 2-qubit gates also increase with $p$, which can lead to a decrease in performance on quantum hardware. To keep the execution time under the set limit, we will only run the experiments for $p=3$, at $K=4$: the combination the provided angles were trained for.

Your results will be graded against a reference set of results. To get the best quality results from your hardware execution, consider the following (non-exhaustive) set of tweaks to your workflow. 

1. Consider doing your analysis with a tailored set of error suppression techniques and select the number of shots appropriately.
2. Do your own training of the QAOA parameters: you can train directly on the hardware, or classically. Note that at $K = 4$ a classical simulator such as the MPS method of [Qiskit-Aer](https://github.com/Qiskit/qiskit-aer) becomes very slow on a laptop. You may have to truncate the maximum bond dimension or experiment with other ways of training.
3. You could also consider warm starting your QAOA, following the techniques described in [this tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/warm-start-qaoa). Note, however, that for this, you would need to change the mixer, the initial state, and then retrain your QAOA parameters.

##### ⚠️ Executing the cell below kicks off a QPU job that takes about 6-8 minutes. Take a second to make sure the circuits in `pubs` are what you want them to be.

In [ ]:
# RUN_DEPTHS and RUN_LEVELS are set at the top of the notebook.

## Set number of shots
## SHOTS = YOUR CODE HERE

jobs = {p: sampler.run(pubs[p], shots=SHOTS) for p in RUN_DEPTHS}
print({p: job.job_id() for p, job in jobs.items()})

In [ ]:
# Ungraded (but good practice)
## Write code to monitor the status of the job
## YOUR CODE HERE

Once the jobs finish running, run the cell below to obtain the hardware results and compute the Pareto front over every shot of every weight setting. The CVaR constraint is imposed on that in the cells that follow. 

In [ ]:
res = {p: jobs[p].result() for p in RUN_DEPTHS}
slot_of = {m: s for s, m in enumerate(weight_list)}

measured = {}
for p in RUN_DEPTHS:
    table_of = {level: src.load_slopes(p, level) for level in pub_levels}
    for q, level in enumerate(pub_levels):
        fields = list(res[p][q].data.keys())
        packed = np.asarray(getattr(res[p][q].data, fields[0]).array, dtype=np.uint8)

        rows = [table_of[level][m] for m in weight_list]
        bits_of = lambda m, pk=packed: src.unpack_bits(pk[slot_of[m]])
        measured[(p, level)] = src.pooled_front(portfolios[level], rows, bits_of)

<div class="alert alert-block" style="background-color:#1b5e20; color:#ffffff; border-color:#1b5e20;">

### **Exercise 2.1: Analyzing the QPU execution; Points [30/100]**

</div>

#### Post-processing the results

Every shot is a candidate portfolio, and each one corresponds to one point in the $(\text{variance}, \text{return})$ plane. Rather than keeping only the best shot of each weight setting, we pool the shots of all the weight settings and keep the ones that are non-dominated (see definition below). 

We say that point $i$ in the return-variance plane is **dominated** when some point $j$ is at least as good on both objectives,

$$\mu^\top x^{(j)} \ \ge\ \mu^\top x^{(i)} \qquad\text{and}\qquad x^{(j)\top} \Sigma\, x^{(j)} \ \le\ x^{(i)\top} \Sigma\, x^{(i)},$$

with at least 1 of the 2 strict. What survives is the measured Pareto front and that is what we want to obtain: not 1 portfolio but the set of portfolios where no
more return is available without more variance. 

These fronts are what we will plot below.

##### Comparison with classical references.

To assess the quality of our hardware results, we will compare them against 3 classical reference solutions. 

1. The first is solutions, encoded in `eps_front` and `eps_bits` below, obtained using the [CPLEX solver](https://www.ibm.com/products/ilog-cplex-optimization-studio/cplex-optimizer) whereby for each return floor $\varepsilon$ on a discretized grid of a chosen length, we solve the following constrained optimization problem:

\begin{equation}
\min_{x\in\{0,1\}^{100}} \; x^\top \Sigma x \quad\text{s.t.}\quad \mu^\top x \;\ge\; \varepsilon\,.
\end{equation}

2. `random_pick` is the second baseline: `SHOTS` samples from a uniform random distribution at every weight
setting, pooled into a Pareto front exactly as the hardware shots are. This corresponds to simulating QAOA at $p=0$, which is the uniform superposition over all $2^{100}$ portfolios. It is a baseline for the hardware results, and it is also a baseline for the CPLEX solutions: if the solver cannot find a better portfolio than random, it is not doing its job.

3. The third baseline is encoded by `ws_bits` below and it corresponds to running CPLEX on the weighted sum objective $f$ defined above, once per weight setting. This is the same objective that QAOA is minimizing, so it offers a direct comparison to the hardware results. 

References 1 and 3 are both similar in that they are both obtained using CPLEX, but they are not the same. Method 1 is a sweep over return floors denoted by $\epsilon$ while method 3 is a sweep over weights. The former can reach every point of the Pareto front, and on a grid of `EPS_COUNT` return floors it approximates the full front; the latter only finds a subset of these points: those that lie on the convex hull of the pareto front. Any non-convex (unsupported) stretches of the front are skipped over by method 3.


##### ⚠️ The following cell runs the CPLEX computation and will take about 1-2 minutes on your laptop.

In [ ]:
SEED = 0
EPS_COUNT = (
    300  # number of epsilons, return floors in the sweep,
    # kept small to keep computation time manageable.
)

angle_rows = {
    level: [src.load_slopes(RUN_DEPTHS[0], level)[m] for m in weight_list]
    for level in pub_levels
}


eps_front, eps_bits, ws_bits, random_pick = {}, {}, {}, {}
for level in pub_levels:
    problem, rows = portfolios[level], angle_rows[level]

    multipliers = [src.trained_weights(row) for row in rows]
    ws_bits[level] = np.array(
        [src.weighted_sum_optimum(problem, a, b)[0] for a, b in multipliers]
    )

    eps_front[level] = src.epsilon_constraint_front(problem, count=EPS_COUNT)
    eps_bits[level] = eps_front[level]["bits"][eps_front[level]["keep"]]

    # A FRESH pool per weight setting, so the control draws the same TOTAL number of
    # samples the device did: 20 settings x SHOTS each.
    def coin_flips(m, n=problem.n, level=level):
        return np.random.default_rng((SEED, level, int(m))).random((SHOTS, n)) < 0.5

    random_pick[level] = src.pooled_front(problem, rows, coin_flips)
references = {
    level: {src.WS_LABEL: ws_bits[level], src.EPS_LABEL: eps_bits[level]}
    for level in pub_levels
}

Hypervolume - A Metric for Quantifying the Quality of a Pareto Front

The hypervolume is a metric for quantifying the quality of a Pareto front. For a multiobjective optimization problem involving two objective functions, the hypervolume is defined as the area beneath the determined Pareto front with respect to a reference point (for more than two objective functions this area becomes a volume).

Generically speaking, a Pareto front with a larger hypervolume represents better quality solutions than a front with a smaller hypervolume.

Let us now compare the quality of our solutions obtained from the quantum hardware to the classical reference solutions using the hypervolume metric. To compute the hypervolume, we will use $(return, variance) = (0, \mathbf{1}^{T}\Sigma \mathbf{1})$ (the worst possible return and the largest possible variance) as the reference point.

In [ ]:
depth_summaries = {
    level: {p: measured[(p, level)] for p in RUN_DEPTHS} for level in pub_levels
}

comparison = {}
for level in pub_levels:
    comparison[level] = src.pareto_depths(
        portfolios[level],
        depth_summaries[level],
        cplex_bits=references[level],
        random_summary=random_pick[level],
        title=f"The pareto fronts at K={level} showing the number of non-dominated points",
        show_points=True,
    )

Let's plot the same results as above but this time as bar plots showing the hypervolume volumes. Recall that a hypervolume is a measure of the quality of a Pareto front, and a larger hypervolume indicates a better Pareto front. 

In [ ]:
labels = [f"p={p}" for p in RUN_DEPTHS] + ["random", src.WS_LABEL, src.EPS_LABEL]
hv = {label: {} for label in labels}
for level in pub_levels:
    problem = portfolios[level]
    ref = src.reference_point(problem)
    for p in RUN_DEPTHS:
        hv[f"p={p}"][level] = src.hypervolume(
            *src.summary_points(measured[(p, level)]), ref
        )
    hv["random"][level] = src.hypervolume(*src.summary_points(random_pick[level]), ref)
    for label, bits in references[level].items():
        values = src.selection_values(problem, np.asarray(bits, dtype=bool))
        hv[label][level] = src.hypervolume(
            values["expected_return"], values["variance"], ref
        )

fraction = {
    label: {
        level: value / src.box_volume(portfolios[level])
        for level, value in item.items()
    }
    for label, item in hv.items()
}
hypervolume_bars = src.hypervolume_figure(
    fraction,
    title=f"p={tuple(RUN_DEPTHS)}: hypervolume against the box "
    r"$[0, \sum_i \mu_i] \times [0, \mathbf{1}^{T} \Sigma \mathbf{1}]$, per sparsity level",
)

You should see that QAOA convincingly outperforms the random baseline and is competitive with the CPLEX methods. A word of caution on the hypervolumes reported here though: for the quantum execution, the Pareto front is computed from all the shots obtained from the hardware, and not just the best shot per weight setting. This can lead to a denser Pareto front and therefore, the hypervolume from the quantum execution can be "inflated" compared to that of the Weighted Sum CPLEX.

This should still be seen as a strength of the quantum workflow: it is able to find a larger number of pareto optimal portfolios essentially because of its sampling nature.

#### Imposing the CVaR constraint.

Let us now turn to the second step in our algorithm, which is to filter out solutions on the Pareto front, based on whether they satisfy the CVaR constraint.
If the loss of a selection is taken
as normal with mean $-\mu^\top x$ and variance $x^\top \Sigma x$, its **conditional value at
risk** at confidence $\alpha$, the mean loss over the worst $1-\alpha$ of outcomes, is

$$\mathrm{CVaR}_\alpha(x) \;=\; -\,\mu^\top x \;+\; k_\alpha \sqrt{x^\top \Sigma x},
\qquad k_\alpha \;=\; \frac{\varphi\!\left(\Phi^{-1}(\alpha)\right)}{1-\alpha}\,,$$

where $\Phi$ is the cumulative distribution function of the standard normal distribution and $\varphi$ is its probability density function. The values of $k_\alpha$ can be evaluated using this information and are given in the table below:

| $\alpha$ | 0.90 | 0.95 | 0.99 |
|---|---|---|---|
| $k_\alpha$ | 1.7550 | 2.0627 | 2.6652 |

Note on CVaR: CVaR is strictly decreasing in the return and strictly
increasing in the variance, so a dominated portfolio can never minimise it — if $x^{(j)}$
beats $x^{(i)}$ on both objectives it beats it on CVaR too. The Pareto front therefore
already contains the CVaR winner of every $\alpha$ at once, and checking for the CVaR constraint  costs a scan over the already sampled portfolios rather than a new quantum run. 

In [ ]:
CVAR_ALPHAS = src.CVAR_ALPHAS  # (0.90, 0.95, 0.99)

CVAR_MODELS = {
    "as optimised (shift, nonempty)": dict(include_shift=True, min_assets=1),
}

cvar_rows = {
    (name, level): src.cvar_table(
        portfolios[level],
        depth_summaries[level],
        cplex_bits=references[level],
        random_summary=random_pick[level],
        alphas=CVAR_ALPHAS,
        **options
    )
    for name, options in CVAR_MODELS.items()
    for level in pub_levels
}

In [ ]:
# CVaR is a loss, so a SHORTER bar is better
CVAR_MODEL_SHOWN = "as optimised (shift, nonempty)"

cvar_bars = {}
for level in pub_levels:
    cvar_bars[level] = src.cvar_figure(
        cvar_rows[(CVAR_MODEL_SHOWN, level)],
        title=f"K={level}: best CVaR per solution set",
    )

Let us now analyze the quality of the solutions obtained from the quantum hardware and compare them to the classical reference solutions, after imposing the CVaR constraint. We will do this for 3 different confidence levels $\alpha = 0.90, 0.95, 0.99$, with $0 \leq c \leq 10$.

We will then do 2 comparisons: first, we will compare how many valid portfolios are found by each method for each confidence level $\alpha$ and for a range of values of $c$. Second, we will compare the best return found by each method for each confidence level $\alpha$ and for a fixed value of $c$.

In [ ]:
C_GRID = np.linspace(0.0, 10.0, 201)

cvar_budget = {
    level: src.cvar_constraint_table(
        portfolios[level],
        depth_summaries[level],
        C_GRID,
        alphas=CVAR_ALPHAS,
        cplex_bits=references[level],
        random_summary=random_pick[level],
        **CVAR_MODELS[CVAR_MODEL_SHOWN],
    )
    for level in pub_levels
}


# 1. how many portfolios each method offers at each budget
budget_counts = {}
for level in pub_levels:
    budget_counts[level] = src.cvar_constraint_panels(
        portfolios[level],
        depth_summaries[level],
        C_GRID,
        quantity="count",
        alphas=CVAR_ALPHAS,
        cplex_bits=references[level],
        random_summary=random_pick[level],
        title=f"K={level}: portfolios admitted by a CVaR budget $c$",
        **CVAR_MODELS[CVAR_MODEL_SHOWN],
    )

In [ ]:
# 2. the constrained optimum: the best return available under each budget
budget_returns = {}
for level in pub_levels:
    budget_returns[level] = src.cvar_constraint_panels(
        portfolios[level],
        depth_summaries[level],
        C_GRID,
        quantity="max_return",
        alphas=CVAR_ALPHAS,
        cplex_bits=references[level],
        random_summary=random_pick[level],
        title=f"K={level}: best return available under a CVaR budget $c$",
        **CVAR_MODELS[CVAR_MODEL_SHOWN],
    )

 All CVaR curves saturate at a sufficiently high value of c. Let's compare them at an intermediate value. We pick c=1.

In [ ]:
BUDGETS = [1.0]

methods = []
for label, _alpha in cvar_budget[pub_levels[0]]:
    if label not in methods:
        methods.append(label)
at = [int(np.argmin(np.abs(C_GRID - c))) for c in BUDGETS]

for level in pub_levels:
    print(f"K={level}: best return available under a CVaR budget c")
    for alpha in CVAR_ALPHAS:
        print(f"  alpha={alpha:g}")
        print(f"    {'c':>5}" + "".join(f"{m:>21}" for m in methods))
        for c, index in zip(BUDGETS, at):
            row = ""
            for label in methods:
                value = cvar_budget[level][(label, float(alpha))]["max_return"][index]
                row += f"{value:>21.4f}" if np.isfinite(value) else f"{chr(8212):>21}"
            print(f"    {c:>5}" + row)
        print()

Note that in the legend "of $<number>$" denotes the total number of candidate non-dominated portfolios obtained by each method.

In [ ]:
# Store results as tables
from pathlib import Path

RESULTS_FILE = Path("results.txt")
CVAR_BOUND = 1.0

blocks = []

for level in pub_levels:
    rows = [
        [
            row["curve"],
            row["nondominated"],
            row["hypervolume"],
            f"{row['hv_fraction']:.2%}",
        ]
        for row in src.pareto_depth_table(
            portfolios[level],
            depth_summaries[level],
            cplex_bits=references[level],
            random_summary=random_pick[level],
        )
    ]
    blocks.append(
        src.results_table(
            f"Pareto front and hypervolume, K={level}, p={tuple(RUN_DEPTHS)}",
            ["method", "nondominated", "hypervolume", "hv / box"],
            rows,
        )
    )

# The methods, in the order every CVaR table lists them.
methods_cvar = []
for row in cvar_rows[(CVAR_MODEL_SHOWN, pub_levels[0])]:
    if row["curve"] not in methods_cvar:
        methods_cvar.append(row["curve"])

# the best CVaR each solution set contains. CVaR is a loss, so lower is better.
best_cvar = {
    (level, row["curve"], row["alpha"]): row
    for level in pub_levels
    for row in cvar_rows[(CVAR_MODEL_SHOWN, level)]
}
for level in pub_levels:
    rows = [
        [method] + [best_cvar[(level, method, float(a))]["cvar"] for a in CVAR_ALPHAS]
        for method in methods_cvar
    ]
    blocks.append(
        src.results_table(
            f"Best CVaR available at K={level}",
            ["method"] + [f"alpha={a:g}" for a in CVAR_ALPHAS],
            rows,
        )
    )
at = int(np.argmin(np.abs(C_GRID - CVAR_BOUND)))
for level in pub_levels:
    rows = [
        [method]
        + [
            cvar_budget[level][(method, float(a))]["max_return"][at]
            for a in CVAR_ALPHAS
        ]
        for method in methods_cvar
    ]
    blocks.append(
        src.results_table(
            f"Best return with the CVaR bound set to c={C_GRID[at]:g}, K={level}",
            ["method"] + [f"alpha={a:g}" for a in CVAR_ALPHAS],
            rows,
        )
    )

# This allows you to save these numbers locally.    
RESULTS_FILE.write_text(
    f"Results for {RESULTS_FILE.parent.resolve().name}\n"
    f"levels K={tuple(pub_levels)}, depths p={tuple(RUN_DEPTHS)}, "
    f"{len(weight_list)} weight settings, {SHOTS:,} shots per setting\n\n"
    + "\n".join(blocks)
)
print(RESULTS_FILE.read_text())

Here as well, you should see that QAOA convincingly outperforms the random baseline and is competitive with (and in some regimes outperforms) the CPLEX methods.
All in all, as you should see, the quantum hardware results are competitive in quality with the classical reference solutions. This is a promising result, and it shows that quantum computers can be used to solve constrained multi-objective optimization problems at scale, providing solutions that are competitive with state-of-the-art classical methods in quality.

As quantum algorithms and hardware continue to develop, we expect quantum computers to outperform (in cost, quality, or both) classical workflows in the near future.

<div class="alert alert-block alert-info">

### **Check your results against the reported ones below (30 pts total).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex2_1

# The 3 tables above are rebuilt from your own measured portfolios: 22.5 pts for
# matching the reported results, and the rest for beating them by more than 1%.
grade_lab_qamoo_ex2_1(portfolios, depth_summaries, references, random_pick)

<div class="alert alert-block" style="background-color:#1b5e20; color:#ffffff; border-color:#1b5e20;">

### **Exercise 2.2: Going deeper; Points [20/100]**

</div>

The hardware circuits so far were built at $p = 3$ at $K = 4$, with the angles read from
`data/angles/angles_p3.csv`. Does more depth always help when doing multiobjective optimization? Let's find out.

To keep the re-training manageable, this exercise moves to $K = 1$. At $K = 1$ we provide trained $p = 3$ angles for 3 weight settings, in `data/angles/angles_k1_p3.csv`. Your task is to **train $p = 4$ at $K=1$ yourself** at
these weight settings (listed below in `DEEPER_WEIGHTS`), run the circuits on hardware, and then analyze the results. Your $p = 4$ run is compared against a reference $p = 3$ run on 2 numbers: the hypervolume of its front, and the best return available under a CVaR budget, `DEEPER_CVAR_BOUND = 0.1` at
`DEEPER_ALPHA = 0.9`.


The cell below is a sketch of the pipeline, not fully working code. If you choose to, you can also completely ignore the snippet below and write your own code, using other tools that you are familiar with.

##### ⚠️ Classical training for this part can take 5-10 minutes on a laptop.

In [ ]:
#  --  a SKETCH.

import csv

# Let DEEPER_WEIGHTS, DEEPER_ALPHA and DEEPER_CVAR_BOUND remain unchanged below.
# We will run only 3 weight settings.
DEEPER_LEVEL = 1
DEEPER_DEPTH = 4
DEEPER_WEIGHTS = (65, 70, 75)
DEEPER_ALPHA = 0.90
DEEPER_CVAR_BOUND = 0.1
DEEPER_SHOTS = ## SET THE NUMBER OF SHOTS
p4_weight_list = list(DEEPER_WEIGHTS)
TRAIN_SHOTS = ## YOUR CODE HERE

# Instantiate a classical simulator for training.
simulator = ## YOUR CODE HERE

portfolios[DEEPER_LEVEL] = src.Portfolio(DEEPER_LEVEL)
problem = portfolios[DEEPER_LEVEL]

# The provided trained p=3 angles at K=1, 1 row per weight setting of DEEPER_WEIGHTS.
p3_rows = {}
with open("data/angles/angles_k1_p3.csv", newline="") as handle:
    for record in csv.DictReader(handle):
        p3_rows[int(record["weight_index"])] = {
            name: float(v) for name, v in record.items()
        }

def bind_angles(circuit, a, b, betas, us, kappa):
    "Bind the 2 weights and the 2p angles. gamma = u / kappa."
    binding = {"cW[0]": float(a), "cW[1]": float(b)}
    for layer, (beta, u) in enumerate(zip(betas, us)):
        binding[f"beta[{layer}]"] = float(beta)
        binding[f"gamma[{layer}]"] = float(u) / kappa
    return circuit.assign_parameters(
        {q: binding[q.name] for q in circuit.parameters}, inplace=False
    )


# ---- 1. the objective to train against ------------------------------------------
# Build expectation values using sampled bitstrings
# Feel free to adjust the number of shots
def objective(params, problem, circ, a, b, kappa, shots=512) -> float:
    p = len(params) // 2
    bound = bind_angles(circ, a, b, params[:p], params[p:], kappa)

    " YOUR CODE HERE"

# ---- 2. train the angles of the p=4 circuit ---------------------------------
# `row` is the provided p=3 row of 1 weight setting: its cW0, cW1
#  and divisor_kappa give the weights and the scale of the cost operator respectively.
#  Train the angles from scratch.

def train_p4(problem, circuit, row) -> tuple[list[float], list[float]]:

    "YOUR CODE HERE"



In [ ]:
# ---- train p=4 at K=1 and assemble one PUB --------------------------------
parametric = build_parametric(problem, DEEPER_DEPTH, pre_swap=True)
isa = pm_informed.run(parametric)

# Build a list of parameters of the parametric circuit
par_names = ## YOUR CODE HERE
values = np.empty((len(p4_weight_list), len(par_names)), dtype=np.float64)
deep_angles = {}

for slot, m in enumerate(p4_weight_list):
    row = p3_rows[m]
    betas, gammas = train_p4(problem, parametric, row)
    deep_angles[m] = (betas, gammas)

    named = {"cW[0]": row["cW0"], "cW[1]": row["cW1"]}
    for layer in range(DEEPER_DEPTH):
        named[f"beta[{layer}]"] = float(betas[layer])
        named[f"gamma[{layer}]"] = float(gammas[layer])
    values[slot] = [named[name] for name in par_names]

p4_depth = isa.depth(lambda x: len(x.qubits) == 2)
print(
    f"K={DEEPER_LEVEL}, p={DEEPER_DEPTH}: trained {len(p4_weight_list)} weight settings, "
    f"transpiled 2q depth {p4_depth}, {isa.num_nonlocal_gates()} 2q gates"
)
## Build p4_pubs, a list holding the 1 PUB (isa, values). Make sure the parameters are
#  assigned in the right lexical order.
## YOUR CODE HERE

# what the grader reads, keyed by level as in exercise 2.1
p4_angles = {DEEPER_LEVEL: deep_angles}
p4_depths = {DEEPER_LEVEL: p4_depth}


##### ⚠️ The QPU job in the cell below runs the $p = 4$ PUB at $K = 1$, and takes 1-2 minutes.

In [ ]:
# ---- 4. run the p=4 PUB (with your choice of error mitigation) -----------------
## job_p4 = ## YOUR CODE HERE
## YOUR CODE HERE

In [ ]:
result_p4 = ## YOUR CODE HERE (retrieve job result)
slot_of = {m: s for s, m in enumerate(p4_weight_list)}
rows = [
    {"weight_index": m, "weight_return": m / 99, "weight_variance": 1 - m / 99}
    for m in p4_weight_list
]

field = list(result_p4[0].data.keys())[0]
packed = np.asarray(getattr(result_p4[0].data, field).array, dtype=np.uint8)
bits_of = lambda m, pk=packed: src.unpack_bits(pk[slot_of[m]])
p4_front = src.pooled_front(problem, rows, bits_of)
print(f"K={DEEPER_LEVEL}, p={DEEPER_DEPTH}: {len(p4_front)} front points")

# what the grader reads, keyed by level and depth
p4_summaries = {DEEPER_LEVEL: {DEEPER_DEPTH: p4_front}}


In [ ]:
table = src.cvar_constraint_table(
    problem,
    {DEEPER_DEPTH: p4_front},
    np.array([DEEPER_CVAR_BOUND]),
    alphas=(DEEPER_ALPHA,),
    include_shift=True,
    min_assets=1,
)
best = table[(f"p={DEEPER_DEPTH}", DEEPER_ALPHA)]["max_return"][0]
shown = f"{best:.4f}" if np.isfinite(best) else "none admitted"
ret, variance = src.summary_points(p4_front)
hv = src.hypervolume(ret, variance, src.reference_point(problem)) / src.box_volume(
    problem
)
print(
    f"K={DEEPER_LEVEL}, p={DEEPER_DEPTH}: hypervolume {hv:.2%} of the box, "
    f"best return at CVaR_{DEEPER_ALPHA:g} <= {DEEPER_CVAR_BOUND:g} is {shown}"
)

<div class="alert alert-block alert-info">

### **Check your $p=4$ results below (20 pts total).**

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_qamoo_ex2_2

# Your p=4 run is compared against the reference p=3
# run at the SAME 3 weight settings.
grade_lab_qamoo_ex2_2(portfolios, p4_summaries, p4_angles=p4_angles, p4_depths=p4_depths)

© IBM Corp. 2026